<a href="https://colab.research.google.com/github/goutham3010/Hospital-Readmission-Prediction-System/blob/main/05_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL READMISSION PREDICTION SYSTEM
## NOTEBOOK 05: MODEL EVALUATION
Comprehensive evaluation of trained classification models using Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix, and ROC Curve analysis.


### 1. IMPORT LIBRARIES


In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

print("Libraries imported successfully.")


### 2. LOAD DATASET


In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

# Auto-detect dataset path (Google Colab / Local repo)
possible_paths = [
    "/content/hospital_readmission_dataset.csv",
    "hospital_readmission_dataset.csv",
    "../data/raw/hospital_readmission_dataset.csv",
    "data/raw/hospital_readmission_dataset.csv"
]

data_path = next((p for p in possible_paths if os.path.exists(p)), "/content/hospital_readmission_dataset.csv")

df = pd.read_csv(data_path)

print(f"Dataset loaded from: {data_path}")
print("Original dataset shape:", df.shape)
display(df.head())


### 3. DATA CLEANING


In [ ]:
# ============================================================
# 3. DATA CLEANING
# ============================================================

# Remove duplicate records
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

# Remove unnecessary / potentially leakage-related columns
columns_to_drop = [
    "patient_id",
    "admission_date",
    "readmission_risk_score"
]

# Drop only columns that exist
columns_to_drop = [
    col for col in columns_to_drop
    if col in df.columns
]

df = df.drop(columns=columns_to_drop)

print("\nRemoved columns:")
print(columns_to_drop)


### 4. SEPARATE FEATURES AND TARGET


In [ ]:
# ============================================================
# 4. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=["label"])
y = df["label"]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


### 5. IDENTIFY FEATURE TYPES


In [ ]:
# ============================================================
# 5. IDENTIFY FEATURE TYPES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


### 6. TRAIN / TEST SPLIT


In [ ]:
# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))


### 7. PREPROCESSING PIPELINES


In [ ]:
# ============================================================
# 7. PREPROCESSING PIPELINES
# ============================================================

# Numerical preprocessing
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# Combine pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline built successfully.")


### 8. DEFINE MODELS


In [ ]:
# ============================================================
# 8. DEFINE MODELS
# ============================================================

models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            max_depth=6,
            class_weight="balanced",
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
}

print(f"Models initialized: {list(models.keys())}")


### 9. TRAIN AND EVALUATE EACH MODEL


In [ ]:
# ============================================================
# 9. TRAIN AND EVALUATE EACH MODEL
# ============================================================

results = {}

trained_models = {}

for model_name, model in models.items():

    print("\n")
    print("=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    # Create complete pipeline
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    # Train
    pipeline.fit(
        X_train,
        y_train
    )

    # Predictions
    y_pred = pipeline.predict(X_test)

    # Probability of class 1
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    # Store results
    results[model_name] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    trained_models[model_name] = pipeline

    # Print results
    print("\nAccuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "Not Readmitted",
                "Readmitted"
            ],
            zero_division=0
        )
    )


### 10. CREATE MODEL COMPARISON TABLE


In [ ]:
# ============================================================
# 10. CREATE MODEL COMPARISON TABLE
# ============================================================

results_df = pd.DataFrame(
    results
).T

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

print("\n")
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

display(
    results_df.round(4)
)


### 11. VISUALIZE MODEL PERFORMANCE


In [ ]:
# ============================================================
# 11. VISUALIZE MODEL PERFORMANCE
# ============================================================

ax = results_df[
    [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(
    "Hospital Readmission Model Performance Comparison"
)

plt.xlabel("Model")
plt.ylabel("Score")

plt.ylim(0, 1)

plt.xticks(rotation=0)

plt.legend(
    loc="lower right"
)

plt.tight_layout()

plt.show()


### 12. SELECT BEST MODEL


In [ ]:
# ============================================================
# 12. SELECT BEST MODEL
# ============================================================

# For this healthcare project, use F1-score
# as the initial selection criterion.

best_model_name = results_df.index[0]

best_model = trained_models[
    best_model_name
]

print("\n")
print("=" * 70)
print("BEST MODEL")
print("=" * 70)

print(
    "Selected Model:",
    best_model_name
)

print(
    "F1 Score:",
    round(
        results_df.loc[
            best_model_name,
            "F1 Score"
        ],
        4
    )
)


### 13. CONFUSION MATRIX FOR BEST MODEL


In [ ]:
# ============================================================
# 13. CONFUSION MATRIX FOR BEST MODEL
# ============================================================

best_predictions = best_model.predict(
    X_test
)

cm = confusion_matrix(
    y_test,
    best_predictions
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Not Readmitted",
        "Readmitted"
    ],
    yticklabels=[
        "Not Readmitted",
        "Readmitted"
    ]
)

plt.title(
    f"Confusion Matrix - {best_model_name}"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

plt.show()


### 14. ROC CURVES FOR ALL MODELS


In [ ]:
# ============================================================
# 14. ROC CURVES FOR ALL MODELS
# ============================================================

plt.figure(figsize=(8, 6))

for model_name, pipeline in trained_models.items():

    probabilities = pipeline.predict_proba(
        X_test
    )[:, 1]

    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )

    auc_score = roc_auc_score(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC={auc_score:.3f})"
    )

# Random classifier baseline
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curves - Model Comparison"
)

plt.legend()

plt.tight_layout()

plt.show()


### 15. SAVE BEST MODEL


In [ ]:
# ============================================================
# 15. SAVE BEST MODEL
# ============================================================

save_model_path = "best_readmission_model.pkl" if not os.path.exists("/content") else "/content/best_readmission_model.pkl"

joblib.dump(
    best_model,
    save_model_path
)

# Also save into models/ folder if available
if os.path.exists("models") or os.path.exists("../models"):
    target_dir = "models" if os.path.exists("models") else "../models"
    joblib.dump(best_model, os.path.join(target_dir, "best_readmission_model.joblib"))

print(
    f"\nBest model saved successfully to: {save_model_path}"
)


### 16. SAVE MODEL COMPARISON


In [ ]:
# ============================================================
# 16. SAVE MODEL COMPARISON
# ============================================================

save_csv_path = "model_evaluation_results.csv" if not os.path.exists("/content") else "/content/model_evaluation_results.csv"

results_df.round(4).to_csv(
    save_csv_path
)

print(
    f"Model evaluation results saved successfully to: {save_csv_path}"
)


### 17. FINAL EVALUATION SUMMARY


In [ ]:
# ============================================================
# 17. FINAL EVALUATION SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("MODEL EVALUATION COMPLETED")
print("=" * 70)

print(
    f"""
Best Model      : {best_model_name}

Accuracy        : {results_df.loc[best_model_name, 'Accuracy']:.4f}
Precision       : {results_df.loc[best_model_name, 'Precision']:.4f}
Recall          : {results_df.loc[best_model_name, 'Recall']:.4f}
F1 Score        : {results_df.loc[best_model_name, 'F1 Score']:.4f}
ROC-AUC         : {results_df.loc[best_model_name, 'ROC-AUC']:.4f}

The model was evaluated using an unseen test set.
"""
)

# ============================================================
# NEXT STEP
# ============================================================

print("""
Next steps:

06 - Hyperparameter Tuning
07 - Explainable AI / Feature Importance
08 - Final Model Saving
09 - Streamlit Application
10 - LLM Integration
11 - RAG Integration
""")
